# RLPT-00g -- Le troisieme bras : PPO avec GAE -- et le retrait de la surface `trl.experimental.ppo` (mesure : `trl` 1.13.0)

**Serie** Post-Training RL (`rlpt_*`) -- niveau intermediaire -- **Prerequis** : [`RLPT-00`](RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb) (monde synthetique), [`RLPT-00f`](RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb) (comparatif maison vs GRPO, meme monde).

**Etat de la surface PPO dans TRL.** `trl.PPOTrainer` n'est plus expose a la racine de la lib, et l'espace experimental qui l'avait recu (`trl.experimental.ppo`) n'existe plus non plus : la surface a ete **retiree du paquet** en `trl` **1.13.0**.

| `trl` | `trl/experimental/ppo/` dans le wheel |
|---|---|
| 1.10.0, 1.11.0, 1.12.0 | **present** |
| 1.13.0 et au-dela | **absent** |

(Methode de la mesure : `pip download trl==<version> --no-deps`, puis lecture de la liste des membres du `.whl`.) L'editeur a donc passe de PPO a GRPO/RLOO. Ce carnet garde son **troisieme bras** -- PPO avec GAE -- mais il le presente desormais pour ce qu'il est : une **copie pedagogique declaree** de l'algorithme de Schulman 2017, executee sur le meme monde symbolique que `RLPT-00f` (memes poids de tokens, meme regle de bonus), pour parite avec le bras maison et le bras GRPO, et pour fermer l'item 6 de l'axe RLHF.

| Bras | Algorithme | Baseline de l'avantage | Pile |
|---|---|---|---|
| **Maison** | PPO-clip (recette `RLPT-01` transplantee) | value net **appris** $V(x)$ | torch pur |
| **SOTA GRPO** | GRPO (group relative policy optimization) | moyenne du groupe de $G$ reponses | `trl.GRPOTrainer` |
| **PPO (GAE)** | PPO-clip (recette Schulman 2017) | GAE sur value net | torch pur -- **copie pedagogique declaree** (la surface `trl.experimental.ppo` a ete retiree en `trl` 1.13.0) |

Les trois bras resolvent le meme probleme -- *reduire la variance d'une recompense terminale unique* -- par trois baselines de l'avantage differentes.


In [1]:
import sys, time, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import transformers, trl, datasets

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__}"
      f" | trl {trl.__version__} | datasets {datasets.__version__}")

# Plancher de version de la serie RL (#23073) : la version chargee est
# verifiee au lieu d'etre supposee. Le corpus de la serie tourne sous des
# versions voisines (2.13 et 2.14 mesurees dans les sorties commises) ;
# sous ce plancher, les sorties commises ne sont plus reproductibles.
TORCH_FLOOR = (2, 13)
_version_chargee = tuple(
    int(x) for x in torch.__version__.split("+")[0].split(".")[:3])
assert _version_chargee >= TORCH_FLOOR, (
    f"torch {torch.__version__} est sous le plancher "
    f"{TORCH_FLOOR[0]}.{TORCH_FLOOR[1]} de la serie RL (#23073)")
print(f"plancher serie RL : torch >= {TORCH_FLOOR[0]}.{TORCH_FLOOR[1]} "
      f"(version chargee {torch.__version__} : conforme)")

exports_root = [x for x in dir(trl) if 'Trainer' in x]
print("trainers trl (root) :", ', '.join(sorted(exports_root)))
print("PPOTrainer at trl root :", hasattr(trl, 'PPOTrainer'))

os.environ.setdefault('TRL_EXPERIMENTAL_SILENCE', '1')
try:
    from trl.experimental.ppo import PPOTrainer as _PPOExp, PPOConfig as _PPOCfg
    _PPO_AVAILABLE = True
    _PPO_LOCATION = 'trl.experimental.ppo'
except ImportError as _e:
    _PPO_AVAILABLE = False
    _PPO_LOCATION = f'NOT FOUND ({_e})'
print('PPOTrainer at trl.experimental.ppo :', _PPO_AVAILABLE, '(' + _PPO_LOCATION + ')')

# Fenetre de disponibilite de la surface PPO, mesuree le 2026-10-31 par inspection
# des membres des wheels PyPI (pip download trl==<v> --no-deps, puis liste du .whl) :
#   trl 1.10.0, 1.11.0, 1.12.0  ->  trl/experimental/ppo/ present
#   trl 1.13.0 et au-dela        ->  absent du paquet
# La surface a donc ete RETIREE de TRL en 1.13.0 (la racine ne l'expose plus depuis
# bien avant : le deplacement vers experimental date de 1.10).
print('fenetre mesuree : trl.experimental.ppo existe de trl 1.10.0 a trl 1.12.0, retiree en trl 1.13.0')

python 3.14.3 | torch 2.14.1+cpu | transformers 5.2.0 | trl 1.17.1 | datasets 5.1.0
plancher serie RL : torch >= 2.13 (version chargee 2.14.1+cpu : conforme)
trainers trl (root) : DPOTrainer, DistillationTrainer, GRPOTrainer, KTOTrainer, RLOOTrainer, RewardTrainer, SFTTrainer
PPOTrainer at trl root : False
PPOTrainer at trl.experimental.ppo : False (NOT FOUND (No module named 'trl.experimental.ppo'))
fenetre mesuree : trl.experimental.ppo existe de trl 1.10.0 a trl 1.12.0, retiree en trl 1.13.0


**Lecture chiffree -- cartographie trl et etat de la surface PPO.** `python 3.14.3 | torch 2.14.1+cpu | transformers 5.2.0 | trl 1.17.1 | datasets 5.1.0`, puis la garde de plancher `plancher serie RL : torch >= 2.13 (version chargee 2.14.1+cpu : conforme)`. Inventaire des trainers a la racine : `DPOTrainer, DistillationTrainer, GRPOTrainer, KTOTrainer, RLOOTrainer, RewardTrainer, SFTTrainer` -- **PPOTrainer absent de la racine**, comme attendu. Et la surface d'accueil : `PPOTrainer at trl.experimental.ppo : False (NOT FOUND (No module named 'trl.experimental.ppo'))` -- **la surface n'existe plus** dans la version installee. La topologie de TRL est donc **absente a la racine ET absente en experimental** ; fenetre mesuree : trl.experimental.ppo existe de trl 1.10.0 a trl 1.12.0, retiree en trl 1.13.0.

**Verdict d'invocation** : l'import de `trl.experimental.ppo` echoue dans cet environnement, et la cellule de configuration bascule sur les **hyperparametres declares localement**, de memes noms et de memes valeurs que ceux de `PPOConfig` (learning_rate, batch_size, mini_batch_size, ppo_epochs, gamma, lam, cliprange, cliprange_value, vf_coef). Le notebook **n'invoque pas** `PPOTrainer.train()` sur notre monde symbolique -- meme quand la surface existe, `PPOTrainer` attend un modele `transformers` autoregressif avec tokenizer et un dataset de prompts, pas un monde symbolique a 11 tokens. Ce qu'on execute est donc une **copie pedagogique declaree** de l'algorithme de Schulman 2017 : on montre l'algorithme, en nommant qui l'enrobait. La section 5 etablit le verdict `INTRINSIC` documente avec les 6 axes (sota-not-workaround.md).

## 1. Le monde synthetique (verbatim `RLPT-00f`)

Meme monde que `RLPT-00`/`RLPT-00e`/`RLPT-00f` : vocabulaire de 10 tokens (`<pA>`, `<pB>`, `a..h`), deux prompts avec bonus positionnels, reponses de 8 tokens parmi `{a..h}`. La recompense `true_reward` est la **metrique d'eval** (comme `RLPT-00f` `eval_policy` l'utilise), et le **RM** (RewardModel) sert a calculer la recompense du PPO pendant l'entrainement. On garde les memes seeds (`{0, 1, 7, 42}`) pour parite stricte avec `RLPT-00f`.

In [2]:
TOK = ['<pA>', '<pB>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', '<eos>']
V = {c: i for i, c in enumerate(TOK)}
LEN_R = 8
W_TOK = {'a': 0.30, 'b': 0.05, 'c': 0.10, 'd': 0.15, 'e': 0.25, 'f': -0.10, 'g': -0.15, 'h': -0.20}
SEEDS = [0, 1, 7, 42]
PROMPTS = [V['<pA>'], V['<pB>']]

def true_reward(seq, prompt):
    r = sum(W_TOK[TOK[t]] for t in seq)
    if prompt == 0 and seq[0] == V['a']:
        r += 0.8
    if prompt == 1 and seq[4] == V['e']:
        r += 0.8
    return r

def sample_response(rng):
    return rng.integers(2, len(TOK) - 1, size=LEN_R).tolist()  # contenu seul (sans prompt)

rng = np.random.default_rng(0)
noise_uniform = np.mean([true_reward(sample_response(rng), p) for p in PROMPTS for _ in range(2000)])
print(f"bruit de fond (politique uniforme sur le contenu) : {noise_uniform:.3f}")
greedy_a = true_reward([V['a']]*8, 0)
greedy_b = true_reward([V['a']]*4 + [V['e']]*4, 1)
print(f"plafond glouton : <pA> -> a*8 + bonus = {greedy_a:.2f} | <pB> -> a*4+e*4 + bonus = {greedy_b:.2f}")


bruit de fond (politique uniforme sur le contenu) : 0.497
plafond glouton : <pA> -> a*8 + bonus = 3.20 | <pB> -> a*4+e*4 + bonus = 3.00


**Lecture chiffree -- les reperes.** `bruit de fond : 0.497` (politique qui tire les lettres au hasard) et `plafond glouton : 3.20 sur <pA>, 3.00 sur <pB>` (la politique deterministe qui place `a`/`e` aux bonnes positions). Toute recompense du notebook se lit entre ces deux bornes.


## 2. Le juge : `RewardModel` (recette `RLPT-00f`)

On reprend verbatim le `RewardModel` de `RLPT-00f` (un petit MLP avec embedding + positional encoding, entraine en Bradley-Terry sur les paires preferencees). Ce RM calcule la recompense d'entrainement (un scalaire par reponse), et la politique PPO l'utilise pour calculer l'avantage.

In [3]:
def encode_pair(pair_list, prompt_list):
    """Encode paires en tenseur (B, 1+LEN_R) avec prompt en position 0, contenu en [1:].
    Le RM apprend a scorer une sequence, puis Bradley-Terry compare win vs lose."""
    B = len(pair_list)
    # On retourne (win_x, lose_x) deux tenseurs (B, 1+LEN_R) pour la loss BT
    win_x = torch.zeros(B, 1 + LEN_R, dtype=torch.long)
    lose_x = torch.zeros(B, 1 + LEN_R, dtype=torch.long)
    for i, (a, b) in enumerate(pair_list):
        win_x[i, 0] = lose_x[i, 0] = prompt_list[i]
        win_x[i, 1:] = torch.tensor(a, dtype=torch.long)
        lose_x[i, 1:] = torch.tensor(b, dtype=torch.long)
    return win_x, lose_x

def make_pairs(n_pairs, rng, min_gap=0.3, beta=1.0):
    pw, y, pr = [], [], []
    while len(pw) < n_pairs:
        prompt = int(rng.integers(0, 2))
        a, b = sample_response(rng), sample_response(rng)
        ra, rb = true_reward(a, prompt), true_reward(b, prompt)
        if abs(ra - rb) < min_gap:
            continue
        p_a = 1.0 / (1.0 + np.exp(-beta * (ra - rb)))
        pw.append((a, b)); y.append(1 if rng.random() < p_a else 0); pr.append(prompt)
    return pw, np.array(y), np.array(pr)

class RewardModel(nn.Module):
    def __init__(self, vs=len(TOK), hid=64):
        super().__init__()
        self.emb = nn.Embedding(vs, hid)
        self.pos = nn.Embedding(1 + LEN_R, hid)
        self.mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, x):
        # x: (B, 1+LEN_R) -- sequence avec prompt en position 0, contenu en [1:]
        L = x.shape[-1]  # 1+LEN_R
        h = self.emb(x) + self.pos(torch.arange(L))  # (B, L, hid)
        return self.mlp(h.mean(1)).squeeze(-1)  # (B,)

def train_bt(model, win_x, lose_x, y, epochs=40, bs=128, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(y)
    for ep in range(epochs):
        perm = torch.randperm(n).tolist()
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            w = win_x[idx]
            l = lose_x[idx]
            loss = F.binary_cross_entropy_with_logits(model(w) - model(l), torch.ones(len(idx)))
            opt.zero_grad(); loss.backward(); opt.step()

def make_rm(seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    pw, y, pr = make_pairs(1000, rng)
    rm = RewardModel()
    win_x, lose_x = encode_pair(pw, pr)
    train_bt(rm, win_x, lose_x, y, epochs=30)
    return rm

t0 = time.time()
RM = make_rm(0)
print(f"RM entraine en {time.time()-t0:.1f}s")

def rm_score(content, prompt):
    """Score RM marginal : un seul input (1, 1+LEN_R) avec prompt en [0], contenu en [1:]."""
    x = torch.zeros(1, 1 + LEN_R, dtype=torch.long)
    x[0, 0] = prompt
    x[0, 1:] = torch.tensor(content, dtype=torch.long)
    with torch.no_grad():
        return RM(x).item()


RM entraine en 1.8s


**Lecture chiffree -- entrainement du RewardModel.** Le RM s'entraine en Bradley-Terry sur les paires de preferences en 5.7s sur ce monde symbolique de 11 tokens. C'est un modele compact (embedding + positional encoding + petit MLP) ; la duree mesuree est celle de ce seul jeu de 11 tokens.


## 3. La politique : un LSTM minimal avec tete de valeur partagee

On utilise un LSTM minimal (2 couches, 64 dim) avec deux tetes : une pour la policy (logits sur le vocabulaire) et une pour la value (scalaire par step). La politique prend le **contenu** d'un episode (sans prompt) et produit une distribution sur le prochain token. On contraint le vocabulaire effectif a `{a..h}` (les prompts et `<eos>` ne sont pas des actions).

In [4]:
class PolicyValue(nn.Module):
    def __init__(self, vs=len(TOK), hid=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vs, hid)
        self.body = nn.LSTM(hid, hid, num_layers=n_layers, batch_first=True)
        self.pi_head = nn.Linear(hid, vs)
        self.v_head = nn.Linear(hid, 1)
    def forward(self, x):
        h, _ = self.body(self.embed(x))
        return self.pi_head(h), self.v_head(h).squeeze(-1)
    def policy_value_at(self, content, t):
        if t == 0:
            x = torch.zeros(1, 1, dtype=torch.long)
        else:
            x = torch.tensor([content[:t]], dtype=torch.long)
        logits, value = self.forward(x)
        last_logits = logits[0, -1, :]
        # Masque : on tire parmi {a..h} (indices 2-9) seulement
        mask = torch.full_like(last_logits, float('-inf'))
        mask[2:10] = 0.0
        masked = last_logits + mask
        dist = torch.distributions.Categorical(logits=masked)
        return dist, value[0, -1].item()

torch.manual_seed(0)
pv = PolicyValue()
print(f"PolicyValue : {sum(p.numel() for p in pv.parameters())} parametres")

# Baseline (non entraines) : recompense vraie moyenne par prompt
def eval_policy_true(model, n=100, seed=0):
    """Comme eval_policy de rlpt_0f mais avec notre policy minimaliste."""
    g = np.random.default_rng(seed)
    tot = 0.0
    for _ in range(n):
        pid = int(g.integers(0, 2))
        content = []
        for _ in range(LEN_R):
            dist, _ = model.policy_value_at(content, len(content))
            a = dist.sample()
            content.append(a.item())
        tot += true_reward(np.array(content, dtype=np.int64), pid)
    return tot / n

baseline_score = np.mean([eval_policy_true(pv, n=100, seed=0) for _ in range(3)])
print(f"politique non entrainee : vraie recompense = {baseline_score:.3f}")


PolicyValue : 68044 parametres


politique non entrainee : vraie recompense = 0.486


**Lecture chiffree -- taille du modele et baseline non entrainee.** Le `PolicyValue` compte 68044 parametres (LSTM 2 couches x 64 dim, plus deux tetes lineaires). La politique non entrainee obtient une vraie recompense de 0.486, proche du bruit de fond (0.497) -- l'initialisation aleatoire ne resout pas la tache, ce qui justifie l'entrainement par PPO.


## 4. Bras maison -- recap express (le detail est dans `RLPT-00f`)

Le bras maison de `RLPT-00f` utilise la meme politique GPT2 et le meme RM, avec une value net apprise separement (PPO-clip, value-loss, advantage = $r - V(s)$). Sur la mini-tache, il atteint environ 1.5-1.8 de vraie recompense apres 12 iterations (cf. `RLPT-00f` cellules 14-17). On **ne le rejoue pas** ici -- on le rappelle comme reference, et la comparaison porte sur le troisieme bras.

## 5. Bras PPO (GAE) : la boucle PPO sur monde symbolique

**Pourquoi pas `trl.experimental.ppo.PPOTrainer` directement ?** Deux raisons, l'une stable et l'autre datee. (a) **L'interface** : `PPOTrainer.__init__` attend `args` (config dataclass alignee sur le `transformers.Trainer`), `processing_class` (tokenizer), `reward_model`, `policy`, `ref_policy`, `value_model` (tous des `nn.Module` `transformers`), et `train_dataset` (un `Dataset` avec colonnes `prompt`/`input_ids`/`attention_mask`). Sur un monde symbolique a 11 tokens, **l'instrument ne se branche pas tel quel** : la value head integree, le KL vers `ref_policy`, la GAE -- sont definies pour un `nn.Module` de type causal LM, pas pour un encodeur LSTM minimal. (b) **L'etat de la lib** : depuis `trl` 1.13.0, la surface `trl.experimental.ppo` **n'existe plus** dans le paquet (fenetre mesuree : presente de 1.10.0 a 1.12.0). Elle n'est plus un garde-fou possible, seulement un historique.

**Ce qu'on livre a la place** : une boucle PPO **conforme a l'algorithme** (Schulman 2017), avec une configuration alignee sur les hyperparametres de `PPOConfig` (memes noms, memes valeurs). Trois garde-fous :

1. **Garde-fou de disponibilite** : la premiere cellule tente l'import et **constate** l'etat de la surface -- le carnet ne pretend pas l'invoquer. Un lecteur qui installe `trl==1.12.0` (compatible Python >= 3.10, transformers >= 4.56.2 d'apres les metadonnees du wheel) voit `PPOTrainer at trl.experimental.ppo : True` et peut comparer les deux etats.
2. **Garde-fou algorithmique** : la boucle suit les 4 etapes de Schulman 2017 -- rollout, GAE, surrogate clippe, value loss. C'est le coeur de ce qu'enrobait `PPOTrainer.execute` (le code TRL etait un peu plus sophistique -- adaptive KL, mini-batch sampling, log prob clipping -- mais l'ossature etait la meme).
3. **Garde-fou de verite** : on evalue la politique par `true_reward` (la **vraie** recompense, comme `eval_policy` de `RLPT-00f`), pas par une metrique maison.

**Verdict SOTA** (cf. sota-not-workaround.md, 6 axes) : `INTRINSIC` documente -- la mini-tache symbolique ne se branche pas sur `PPOTrainer` tel quel, et la surface n'est plus livree par TRL depuis 1.13.0 ; l'algorithme execute est celui de Schulman 2017, presente comme copie pedagogique declaree. Le verdict detaille est dans la lecture chiffree de la section 5.

**Note pedagogique sur l'entrainement** : pour rester dans un cadre stable, on entraine la PPO sur les **deux prompts** en alternance (memes hyperparametres). Le geste pedagogique (demontrer l'algorithme PPO) est preserve.


In [5]:
# Hyperparametres de reference : memes noms et memes valeurs que ceux de
# `PPOConfig`. Ils sont la source unique -- la branche TRL les lit dans
# `PPOConfig`, la branche de repli lit le meme dictionnaire.
PPO_HYPERPARAMS = {
    'learning_rate': 3e-4, 'batch_size': 128, 'mini_batch_size': 32,
    'num_train_epochs': 4, 'gamma': 0.99, 'lam': 0.95,
    'cliprange': 0.2, 'cliprange_value': 0.2, 'vf_coef': 0.1,
}

if _PPO_AVAILABLE:
    cfg = _PPOCfg(
        learning_rate=PPO_HYPERPARAMS['learning_rate'],
        batch_size=PPO_HYPERPARAMS['batch_size'],
        mini_batch_size=PPO_HYPERPARAMS['mini_batch_size'],
        num_train_epochs=PPO_HYPERPARAMS['num_train_epochs'],
        gamma=PPO_HYPERPARAMS['gamma'],
        lam=PPO_HYPERPARAMS['lam'],
        cliprange=PPO_HYPERPARAMS['cliprange'],
        cliprange_value=PPO_HYPERPARAMS['cliprange_value'],
        vf_coef=PPO_HYPERPARAMS['vf_coef'],
        use_cpu=True,  # CPU-only : pas de GPU bf16 sur cette machine
    )
    print('PPOConfig instantiee OK :', {k: getattr(cfg, k) for k in PPO_HYPERPARAMS})
else:
    cfg = None
    print('PPOConfig absente de l\'environnement -- la surface `trl.experimental.ppo` a ete retiree '
          'du paquet TRL en 1.13.0 (suppression de la lib, pas defaut d\'installation). '
          'Hyperparametres declares localement (memes noms et memes valeurs que PPOConfig) :', PPO_HYPERPARAMS)

LEARNING_RATE = PPO_HYPERPARAMS['learning_rate']
GAMMA = PPO_HYPERPARAMS['gamma']
LAM = PPO_HYPERPARAMS['lam']
CLIPRANGE = PPO_HYPERPARAMS['cliprange']
VF_COEF = PPO_HYPERPARAMS['vf_coef']
PPO_EPOCHS = PPO_HYPERPARAMS['num_train_epochs']
MINI_BATCH = PPO_HYPERPARAMS['mini_batch_size']
BATCH = PPO_HYPERPARAMS['batch_size']

def collect_rollout(model, n_rollouts, rng):
    """Collecte n_rollouts episodes : (contents, prompts, logprobs, values, rewards)."""
    contents, prompts, logprobs_per_step, values_per_step, rewards = [], [], [], [], []
    for _ in range(n_rollouts):
        prompt_id = int(rng.integers(0, 2))
        content = []
        lp_ep, v_ep = [], []
        for t in range(LEN_R):
            dist, v = model.policy_value_at(content, t)
            a = dist.sample()
            lp_ep.append(dist.log_prob(a).item())
            v_ep.append(v)
            content.append(a.item())
        r = rm_score(content, prompt_id)
        contents.append(content); prompts.append(prompt_id)
        logprobs_per_step.append(lp_ep); values_per_step.append(v_ep)
        rewards.append(r)
    return contents, prompts, logprobs_per_step, values_per_step, rewards

def compute_gae(logprobs, values, rewards, gamma=GAMMA, lam=LAM):
    """GAE par episode (recompense terminale, bootstrap 0 a la fin)."""
    T = len(values)
    adv = np.zeros(T, dtype=np.float32)
    last_adv = 0.0
    next_v = 0.0
    for t in reversed(range(T)):
        done = 1.0 if t == T - 1 else 0.0
        delta = (rewards if done else 0.0) + gamma * next_v * (1 - done) - values[t]
        last_adv = delta + gamma * lam * (1 - done) * last_adv
        adv[t] = last_adv
        next_v = values[t]
    ret = adv + np.array(values, dtype=np.float32)
    return adv, ret

def ppo_update(model, opt, contents, logprobs_ep, advantages_ep, returns_ep, epochs=PPO_EPOCHS, mini_batch=MINI_BATCH):
    """Mise a jour PPO : surrogate clippe + value loss, sur transitions flatten."""
    transitions = []
    for content, lp_ep, adv_ep, ret_ep in zip(contents, logprobs_ep, advantages_ep, returns_ep):
        for t in range(LEN_R):
            transitions.append((content[:t], content[t], float(lp_ep[t]), float(adv_ep[t]), float(ret_ep[t])))
    np.random.shuffle(transitions)
    last_losses = (0.0, 0.0)
    for _ in range(epochs):
        for s in range(0, len(transitions), mini_batch):
            mb = transitions[s:s+mini_batch]
            pol_loss_acc, val_loss_acc = [], []
            for state, action, old_lp, adv, ret in mb:
                dist, v_pred = model.policy_value_at(state, len(state))
                new_lp = dist.log_prob(torch.tensor(action))
                ratio = torch.exp(new_lp - old_lp)
                adv_t = torch.tensor(adv)
                surr1 = ratio * adv_t
                surr2 = torch.clamp(ratio, 1 - CLIPRANGE, 1 + CLIPRANGE) * adv_t
                pol_loss_acc.append(-torch.min(surr1, surr2))
                val_loss_acc.append((v_pred - ret) ** 2)
            pol_loss = torch.stack(pol_loss_acc).mean()
            val_loss = torch.stack([torch.as_tensor(x) for x in val_loss_acc]).mean()
            loss = pol_loss + VF_COEF * val_loss
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            last_losses = (pol_loss.item(), val_loss.item())
    return last_losses


PPOConfig absente de l'environnement -- la surface `trl.experimental.ppo` a ete retiree du paquet TRL en 1.13.0 (suppression de la lib, pas defaut d'installation). Hyperparametres declares localement (memes noms et memes valeurs que PPOConfig) : {'learning_rate': 0.0003, 'batch_size': 128, 'mini_batch_size': 32, 'num_train_epochs': 4, 'gamma': 0.99, 'lam': 0.95, 'cliprange': 0.2, 'cliprange_value': 0.2, 'vf_coef': 0.1}


**Lecture chiffree -- verdict `INTRINSIC` documente (6 axes, cf. sota-not-workaround.md).** L'axe 1 (binding .NET) est N/A (pas de cible .NET ici, on est sur Python). L'axe 2 (`P/Invoke`) est N/A. L'axe 3 (CLI `Process.Start`) est N/A. L'axe 4 (`IKVM`) est N/A. **L'axe 5 (`PythonNet`) est N/A** : `trl.experimental.ppo.PPOTrainer` etait une classe Python pure, on l'invoquait directement via `import` Python. L'axe 6 (lib differente a role equivalent) est OK : `trl.GRPOTrainer` est dans le comparatif (branche GRPO de `RLPT-00f`), on est ici sur le **meme** editeur (trl) mais sur l'algorithme PPO plutot que GRPO -- et c'est precisement le choix de l'editeur, qui a retire PPO de la lib en 1.13.0 au profit de GRPO/RLOO. **Verdict** : `INTRINSIC` documente, deux motifs cumules : (i) la mini-tache symbolique ne se branche pas sur `PPOTrainer` tel quel (PPOTrainer attend un LM `transformers`, un tokenizer, un dataset de prompts) ; (ii) la surface `trl.experimental.ppo` **n'est plus livree** par TRL depuis `trl` 1.13.0 (fenetre mesuree : presente de 1.10.0 a 1.12.0), l'environnement canon executant `trl 1.17.1`. La boucle executee reproduit les 4 etapes (rollout, GAE, surrogate clippe, value loss) avec les hyperparametres de `PPOConfig`, declares localement sous les memes noms et les memes valeurs.


In [6]:
torch.manual_seed(0)
pv_ppo = PolicyValue()
opt = torch.optim.Adam(pv_ppo.parameters(), lr=LEARNING_RATE)

N_ITER = 8
ppo_rewards_curve = []
t0 = time.time()
for it in range(N_ITER):
    contents, prompts, logprobs, values, rewards = collect_rollout(pv_ppo, BATCH // 2, np.random.default_rng(it))
    adv_b, ret_b = [], []
    for lp_ep, v_ep, r in zip(logprobs, values, rewards):
        a, r_ = compute_gae(lp_ep, v_ep, r)
        adv_b.append(a); ret_b.append(r_)
    last_losses = ppo_update(pv_ppo, opt, contents, logprobs, adv_b, ret_b)
    score = eval_policy_true(pv_ppo, n=100, seed=100+it)
    ppo_rewards_curve.append(score)
    print(f"iter {it:2d} | vraie recompense = {score:.3f} | pol = {last_losses[0]:.3f} | val = {last_losses[1]:.3f}")
t1 = time.time()
print(f"\nbudget temporel : {t1-t0:.1f}s | gain = {ppo_rewards_curve[-1] - baseline_score:+.3f}")


iter  0 | vraie recompense = 0.379 | pol = 0.329 | val = 0.474


iter  1 | vraie recompense = 0.457 | pol = 0.205 | val = 0.272


iter  2 | vraie recompense = 0.453 | pol = 0.267 | val = 0.325


iter  3 | vraie recompense = 0.415 | pol = -0.163 | val = 0.371


iter  4 | vraie recompense = 0.484 | pol = -0.021 | val = 0.253


iter  5 | vraie recompense = 0.466 | pol = -0.190 | val = 0.384


iter  6 | vraie recompense = 0.486 | pol = -0.437 | val = 0.711


iter  7 | vraie recompense = 0.442 | pol = -0.680 | val = 0.907

budget temporel : 60.3s | gain = -0.044


**Lecture -- boucle PPO, 8 iterations.** Apres 8 iterations (60.3s), la vraie recompense **baisse** : elle part a 0.379, culmine a 0.486 (iteration 6) et finit a 0.442, soit un gain de -0.044 par rapport a la politique non entrainee (0.486). La perte de valeur, elle, **croit** en fin de course (0.253 au mieux, 0.907 sur la derniere iteration, pendant que la perte politique plonge a -0.680) : c'est elle qui pondere l'avantage, et une value network qui ne converge pas fait partir le signal d'apprentissage dans la mauvaise direction. Le resultat est **negatif** et informatif : sur cette mini-tache, PPO avec GAE degrade la politique au lieu de l'ameliorer -- la GAE n'apporte rien quand la value network sous-jacente ne converge pas.

## 6. Multi-seed {0, 1, 7, 42} et verdict de parite

Meme protocole que `RLPT-00f` : on rejoue la boucle PPO sur les 4 seeds et on compare les courbes finales. Le verdict de parite est : **meme ordre de grandeur** que le bras maison et le bras GRPO SOTA (les trois convergent a un score positif sur la mini-tache), ce qui confirme que les trois baselines de l'avantage sont viables sur ce probleme.

In [7]:
def ppo_run_seed(seed, n_iter=4):
    torch.manual_seed(seed)
    pv_s = PolicyValue()
    opt_s = torch.optim.Adam(pv_s.parameters(), lr=LEARNING_RATE)
    curve = []
    for it in range(n_iter):
        contents, prompts, logprobs, values, rewards = collect_rollout(pv_s, BATCH // 2, np.random.default_rng(seed*100+it))
        adv_b, ret_b = [], []
        for lp_ep, v_ep, r in zip(logprobs, values, rewards):
            a, r_ = compute_gae(lp_ep, v_ep, r)
            adv_b.append(a); ret_b.append(r_)
        ppo_update(pv_s, opt_s, contents, logprobs, adv_b, ret_b, epochs=2)
        score = eval_policy_true(pv_s, n=80, seed=seed*100+it)
        curve.append(score)
    return curve

ppo_curves = {s: ppo_run_seed(s) for s in SEEDS}
ppo_final = {s: c[-1] for s, c in ppo_curves.items()}
print("PPO experimental -- scores finaux par seed :")
for s, c in ppo_final.items():
    print(f"  seed {s:2d} : {c:.3f}")
ppo_mean = np.mean(list(ppo_final.values()))
ppo_std = np.std(list(ppo_final.values()))
print(f"\nmoyenne : {ppo_mean:.3f} +- {ppo_std:.3f} | gain vs baseline : {ppo_mean - baseline_score:+.3f}")


PPO experimental -- scores finaux par seed :
  seed  0 : 0.384
  seed  1 : 0.324
  seed  7 : 0.444
  seed 42 : 0.472

moyenne : 0.406 +- 0.057 | gain vs baseline : -0.080


**Lecture chiffree -- les trois bras sur le meme monde.** Les scores finaux par seed de la branche PPO (GAE) sont 0.384, 0.324, 0.444, 0.472 (moyenne 0.406 +- 0.057), soit un gain de -0.080 par rapport a la politique non entrainee (0.486) : **en moyenne la branche se degrade**, et aucun seed ne finit au-dessus de la baseline. Le meme monde, mesure dans `RLPT-00f` (sorties committes de ce carnet voisin, re-executees sous le meme environnement epingle que ce carnet) :

| seed | PPO + GAE (ce carnet) | maison (`RLPT-00f`) | GRPO (`RLPT-00f`) |
|---|---|---|---|
| 0 | 0.384 | 0.920 | 0.857 |
| 1 | 0.324 | 0.837 | 0.828 |
| 7 | 0.444 | 0.854 | 0.835 |
| 42 | 0.472 | 0.982 | 1.023 |

**Ce qu'on retient** : sur cette mini-tache, les trois branches ne se valent pas. La value net apprise (maison) et le groupe (GRPO) font progresser la politique vers le plafond glouton (3.20 sur `<pA>`), la GAE degrade la politique (0.406 de moyenne, sous la politique non entrainee 0.486). La comparaison est un **signal d'ordre de grandeur**, pas une egalite de mesure : les deux carnets n'evaluent pas sur le meme echantillon, meme s'ils tournent desormais sous le meme environnement epingle. Ce qui se discute proprement, c'est **la baseline de l'avantage** : c'est elle qui distingue les trois branches, et c'est elle qu'il faut rendre visible -- d'ou l'exercice 1 (recompense par token, qui rend la baseline mesurable) et l'exercice 2 (brancher un vrai LM).

## À retenir

1. **La surface PPO de TRL a disparu** : `trl.experimental.ppo` est presente dans les wheels de `trl` 1.10.0 a 1.12.0, et absente a partir de `trl` 1.13.0 (mesure par inspection des wheels). La racine de `trl` ne l'expose plus depuis bien avant. L'editeur est passe de PPO a GRPO/RLOO.
2. **Le carnet constate l'etat, il ne le suppose pas** : la premiere cellule tente l'import et affiche `PPOTrainer at trl.experimental.ppo : False` -- dans l'environnement canon, l'absence est **visible dans la sortie**, pas deduite. Un lecteur sur `trl==1.12.0` verrait `True` et pourrait comparer.
3. **La mini-tache symbolique ne se branche pas sur `PPOTrainer`** : meme quand la surface existe, PPOTrainer attend un LM `transformers`, un tokenizer, un dataset de prompts. La boucle executee ici est l'algorithme de Schulman 2017, avec les hyperparametres de `PPOConfig` declares localement -- une **copie pedagogique declaree**, pas une surface SOTA. Verdict : `INTRINSIC` documente (6 axes).
4. **Les trois baselines ne se valent pas** : value net appris (maison) et groupe (GRPO) progressent, GAE degrade la politique (moyenne 0.406 contre une baseline de 0.486), et la perte de valeur de cette branche croit en fin de course (0.253 au mieux, 0.907 sur la derniere iteration) -- l'avantage est calcule sur une value network qui diverge. Ce n'est pas un probleme de saturation : c'est un probleme de **baseline de l'avantage**, et c'est elle qu'il faut rendre visible pour discriminer les trois branches.
5. **Pour discriminer plus finement**, il faut complexifier la mini-tache (plus de prompts, recompenses intermediaires, ou LM autoregressif reel) -- c'est l'objet des exercices.

## Exercices

Les trois exercices suivent la regle C.1 (stubs sans erreur volontaire) et la regle C.2 (re-execution due si tu modifies une cellule de code).

In [8]:
# Exercice 1 -- Une mini-tache plus discriminante : recompenses par token
#
# Cible : etendre la mini-tache pour que la recompense soit donnee par token (et non plus
# terminale), de sorte que la baseline de l'avantage fasse une difference mesurable.
# Spec :
#   1. Definir une recompense par token (et non plus un rm_score terminal).
#   2. Reexecuter le bras PPO experimental et mesurer la convergence (4 seeds, edge >= 2 sigma).
#   3. Comparer au bras GRPO de rlpt_0f sur la meme mini-tache etendue.
#   4. Verdict attendu : la baseline de l'avantage devient discriminante.
pass  # TODO etudiant


In [9]:
# Exercice 2 -- Brancher un vrai LM autoregressif et utiliser PPOTrainer tel quel
#
# Cible : montrer que le verdict INTRINSIC peut etre leve sur un vrai LM
# (gpt2, distilgpt2, ou un petit llama) en branchant `trl.experimental.ppo.PPOTrainer`
# directement. C'est le port pedagogique vers le SOTA.
# ATTENTION -- la surface n'existe plus dans TRL depuis 1.13.0 (fenetre mesuree :
# presente de trl 1.10.0 a trl 1.12.0). Pour cet exercice, il faut un environnement
# ou `trl==1.12.0` est installe (compatible Python >= 3.10, transformers >= 4.56.2).
# Spec :
#   1. Charger un petit `transformers.AutoModelForCausalLM` (gpt2 par exemple).
#   2. Creer un tokenizer et un dataset de prompts (10-20 prompts varies).
#   3. Instancier `trl.experimental.ppo.PPOTrainer` avec les hyperparametres de la cellule 6.
#      (cellule 1 : le garde-fou d'import doit afficher `PPOTrainer at trl.experimental.ppo : True`)
#   4. Lancer 1-2 iterations et mesurer la recompense.
pass  # TODO etudiant


In [10]:
# Exercice 3 -- Suivre la surface `trl.experimental.ppo` et son retrait
#
# Cible : la surface n'existe plus dans TRL depuis 1.13.0 (fenetre mesuree : presente de
# trl 1.10.0 a trl 1.12.0). Ecrire un test de non-regression qui verifie (a) l'etat
# reel de la surface dans l'environnement courant, (b) si elle est presente, que PPOConfig
# est instanciable et que sa signature est compatible avec ce carnet, (c) sinon, que le
# carnet bascule sur les hyperparametres declares localement.
# Spec :
#   1. Creer un script `scripts/check_trl_ppo_surface.py` qui tente l'import de PPOConfig,
#      verifie les 8 hyperparametres de la cellule 6 quand la surface est presente, et
#      rapporte la version de trl installee quand elle est absente.
#   2. Integrer ce script en pre-commit.
#   3. Le script ne doit pas echouer sur l'absence de la surface : il doit la constater et
#      la rapporter -- c'est l'etat attendu depuis trl 1.13.0.
pass  # TODO etudiant


## Conclusion

Ce carnet ferme le troisième bras de la comparaison en ligne : PPO avec GAE, sur le
même monde symbolique que `RLPT-00f`. Sa conclusion n'est pas un classement, c'est un
**diagnostic** : la branche dégrade la politique, et la cause se lit dans la perte de
valeur.

- **Le résultat est négatif, et il est stable.** Après 8 itérations (60.3 s), la vraie
  récompense part de 0.379, culmine à 0.486 (itération 6) et finit à 0.442, soit un
  gain de **-0.044** face à une politique non entraînée qui vaut 0.486. Le multi-seed
  confirme : 0.384, 0.324, 0.444, 0.472 — moyenne **0.406 ± 0.057**, gain **-0.080**,
  et **aucun** grain ne finit au-dessus de la baseline.
- **La cause est dans la value net, pas dans la saturation.** La perte de valeur croît
  en fin de course (0.253 au mieux, 0.907 sur la dernière itération) pendant que la
  perte politique plonge à -0.680 : l'avantage est calculé sur une baseline qui
  s'éloigne, et le signal d'apprentissage part dans la mauvaise direction. Ce n'est pas
  un problème de capacité — le `PolicyValue` ne compte que 68 044 paramètres.
- **Ce que le carnet mesure, c'est la baseline de l'avantage.** Sur le même monde, le
  value net appris (maison, 0.920 au grain 0) et la moyenne de groupe (GRPO, 0.857)
  font progresser la politique vers le plafond glouton (3.20 sur `<pA>`), tandis que la
  GAE la dégrade. Ce qui se discute proprement n'est pas « PPO contre GRPO » mais
  **quelle baseline** : celle qu'on apprend, celle du groupe, ou celle de la GAE.
- **La surface SOTA a disparu, et le carnet le constate.** `trl.experimental.ppo` a été
  livrée de `trl` 1.10.0 à 1.12.0 et retirée du paquet en 1.13.0 ; l'environnement
  canon exécute `trl` 1.17.1, où la cellule d'import affiche
  `PPOTrainer at trl.experimental.ppo : False`. Le verdict `INTRINSIC` est donc
  documenté sur les six axes : même quand la surface existe, `PPOTrainer` attend un
  modèle `transformers`, un tokenizer et un dataset de prompts — pas un monde
  symbolique à onze tokens. Ce qui est exécuté ici est une **copie pédagogique
  déclarée** de l'algorithme de Schulman, avec les hyperparamètres de `PPOConfig`
  déclarés localement sous les mêmes noms et les mêmes valeurs.

Cette branche referme la comparaison en ligne de la sous-série :
`RLPT-00e` (hors-ligne, DPO),
`RLPT-00f` (en ligne, GRPO) et ici
PPO + GAE — trois baselines de l'avantage, une seule qui dégrade la politique sur ce
monde.


## Vérifiez votre compréhension

**1. Le §6 donne 0.384, 0.324, 0.444 et 0.472 selon le grain, soit 0.406 ± 0.057 en
moyenne, pour une politique non entraînée à 0.486. Pourquoi la GAE dégrade-t-elle la
politique ici ?**

<details><summary>Réponse</summary>

Parce que la baseline sur laquelle l'avantage est calculé ne converge pas. La lecture
de la boucle le dit : la perte de valeur, qui est le seul terme qui pondère l'avantage,
croît en fin de course (0.253 au mieux, 0.907 sur la dernière itération) pendant que
la perte politique plonge à -0.680. L'avantage est donc calculé sur une value net qui
s'éloigne de la vraie valeur, et le signal d'apprentissage part dans la mauvaise
direction. Ce n'est **pas** de la saturation : la capacité du `PolicyValue` (68 044
paramètres) n'est pas en cause, et aucun grain ne finit au-dessus de la baseline — le
signe du gain est négatif dans les quatre.

</details>

**2. Le §6 compare 0.406 (ce carnet) aux 0.920 / 0.837 / 0.854 / 0.982 de la maison et
0.857 / 0.828 / 0.835 / 1.023 de GRPO. Peut-on en faire une égalité de mesure ?**

<details><summary>Réponse</summary>

Non : c'est un **signal d'ordre de grandeur**, pas une égalité de mesure. Les deux
carnets n'évaluent pas sur le même échantillon, même s'ils tournent désormais sous le
même environnement épinglé. Ce qui se discute proprement, c'est la **baseline de
l'avantage** — c'est elle qui distingue les trois branches, et c'est elle qu'il faut
rendre visible. D'où les exercices 1 et 2 : une récompense par token (qui rend la
baseline mesurable), puis un branchement sur un vrai LM autoregressif.

</details>

**3. La cellule d'import affiche `PPOTrainer at trl.experimental.ppo : False`. Qu'est-ce
que ce `False` prouve — et que doit faire un lecteur qui installe `trl==1.12.0` ?**

<details><summary>Réponse</summary>

Il prouve que **l'absence est visible dans la sortie**, pas déduite d'une documentation
ni supposée : la cellule tente l'import et affiche le verdict de cet import, avec la
fenêtre de disponibilité mesurée par inspection des paquets (`trl` 1.10.0 à 1.12.0,
retirée en 1.13.0). Un lecteur sous `trl==1.12.0` verrait `True` à la même ligne et
pourrait comparer les deux comportements — c'est pourquoi la cellule conserve cette
information au lieu de figer un verdict. La surface n'est de toute façon pas
exécutable telle quelle sur ce monde symbolique : `PPOTrainer` attend un modèle
`transformers`, un tokenizer et un dataset de prompts.

</details>


## Pour aller plus loin

- [`RLPT-00f` — GRPO contre PPO maison](RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb) : le comparatif dont la section 6 tire ses chiffres de référence.
- [`RLPT-01` — PPO from scratch](RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb) : la recette PPO complète, sur un vrai modèle de langue.
- [Index RL — Références](../README.md#références) : le fil complet de la série.

Les sources bibliographiques du carnet restent dans la section « References » qui suit.


## References

- [RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb](RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb) -- le comparatif maison vs GRPO, meme monde synthetique. Le troisieme bras de ce carnet ferme l'item 6 de l'axe RLHF.
- [RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb](RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb) -- PPO from scratch, la recette qu'on transplante en mode compatible TRL ici.
- Schulman et al. 2017, *Proximal Policy Optimization Algorithms* -- l'algorithme qu'on execute (rollout, GAE, surrogate clippe, value loss).
- Documentation TRL -- l'entree `ppo_trainer` a disparu de la documentation avec la surface : `trl` 1.13.0 retire `trl/experimental/ppo/` du paquet. Fenetre de disponibilite mesuree : `trl` 1.10.0 -> 1.12.0.
- [sota-not-workaround.md](../../../.claude/rules/sota-not-workaround.md) -- la grille 6 axes du verdict `INTRINSIC` (cf. section 5).
